In [1]:
import json
import re
from nltk.tokenize import word_tokenize,sent_tokenize
from nltk import pos_tag
from tqdm import tqdm
from nltk.corpus import wordnet as wn
from collections import Counter

import warnings
warnings.filterwarnings('ignore')

from nltk.corpus import stopwords
stopword_list = stopwords.words('english')
stopword_list.extend(['``','\'s','\'\'','n\'t'])

def wordnet_hypernyms(token):
    all_hypernyms = []

    word_senses = wn.synsets(token)
    hypernyms = lambda s: s.hypernyms()
    for ws in word_senses:
        hypernyms = [hyp.name() for hyp in list(ws.closure(hypernyms))]
        all_hypernyms.extend(hypernyms)
    return all_hypernyms

def intersection(list1,list2):
    return list(set(list1) & set(list2))
    

def remove_punctuation_and_stopwords(words):
    new_list= []
    for w in words:
        if w.isalnum() and w not in stopword_list:
            new_list.append( w )
    return new_list

In [2]:
json_data = json.load(open('reviews.json'))

In [3]:
words_freq = Counter()

emotions = ['emotion.n.01',
 'feeling.n.01'] 

selected_tags = [ 'JJ',
'JJR',
'JJS ',
'NN',
'NNS',
'RB',
'RBR',
'RBS',
'VB',
'VBD',
'VBG',
'VBN',
'VBP',
'VBZ' ]


for review in tqdm(json_data):
    words = word_tokenize(review['text'].lower())
    words = remove_punctuation_and_stopwords(words)
    tagged_words = pos_tag(words)
    for word,tag in tagged_words:
        if tag in selected_tags:
            if len(intersection(wordnet_hypernyms(word),emotions))>0:
                words_freq.update([word])

                

100%|████████████████████████████████████| 68512/68512 [04:35<00:00, 248.33it/s]


In [4]:
emotion_words = list(words_freq.keys())

emotions_combined = "(" + ")|(".join(emotion_words) + ")"


In [5]:
# Identification
patterns = [
    'ma((de)|(kes?))\s+me\s+((realise)|(discover))'
'identity',
    'self-discovery',
    'identification',
    'identify',
    'revelation',
    'discovery',
    'discover',
    'authentically',
    'closet',
    '((who)|(what))\s+i\s+am',
    'my\s+(own\s+)?journey',
    '(my\s+)?self-((worth)|(esteem)|(hatred)|(importance))',
    'means\s+(\w+\s+){0,3}to\s+me',
    'that\s+i\s+am\s((gay)|(queer))',
    'my\s+(\w+\s+){0,3}personal\s+life'
]



identification_combined = "(" + ")|(".join(patterns) + ")"



In [12]:
all_data = []

for review in tqdm(json_data):
    row = dict()
    review_text = review['text']
    row['id'] = review['book_id']
    if review['type']=='review':
        row['review_id'] = int(review['review_id'])

    
    annotations = []
    
    sentences = sent_tokenize(review_text)
    for sentence in sentences:
        
        if re.search(rf'{identification_combined}',sentence,re.IGNORECASE):
            annotation_dict = dict()
            annotation_dict['sentence'] = sentence
            annotation_dict['reference'] = 'identification'
            annotations.append(annotation_dict)
            
        if re.search(rf'{emotions_combined}',sentence,re.IGNORECASE):
            annotation_dict = dict()
            annotation_dict['sentence'] = sentence
            annotation_dict['reference'] = 'emotional_language'
            annotations.append(annotation_dict)

        if re.search(r'\b((trump)|(republican)|(maga))',sentence,re.IGNORECASE):
            annotation_dict = dict()
            annotation_dict['sentence'] = sentence
            annotation_dict['reference'] = 'trump'
            annotations.append(annotation_dict)
            
        if re.search(r'\bstonewall',sentence,re.IGNORECASE):
            annotation_dict = dict()
            annotation_dict['sentence'] = sentence
            annotation_dict['reference'] = 'stonewall'
            annotations.append(annotation_dict)
            
        if re.search(r'\b((gay)|(lesbian)|(lgbt?q?))\s+((right)|(law))',sentence,re.IGNORECASE):
            annotation_dict = dict()
            annotation_dict['sentence'] = sentence
            annotation_dict['reference'] = 'gay rights'
            annotations.append(annotation_dict)

    if len(annotations)>0:
        row['annotations'] = annotations
    if review['type']=='review':
        all_data.append(row)     

100%|█████████████████████████████████████| 68512/68512 [19:28<00:00, 58.64it/s]


In [13]:
with open('annotations.json','w',encoding='utf-8') as out:
    out.write( json.dumps(all_data, indent=4) )

In [14]:
review_base_url = 'https://www.goodreads.com/review/show/'

out = open('annotations.tsv','w',encoding='utf-8')
out.write('sentence\tclaim\treview\n')
          

for row in all_data:

    review_id = row['review_id']
    if 'annotations' in row:
        for annotation in row['annotations']:
            #print(annotation)
            sentence = annotation['sentence']
            claim = annotation['reference']
            
            review_url = ''
            if re.search('\d',str(review_id)):
                review_url = f'{review_base_url}{review_id}'
            
            out.write(f'{sentence}\t{claim}\t{review_url}\n')

            
out.close()            
    